# Random Forest

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import joblib

from sklearn.linear_model import LogisticRegression


# 1. Load Data
data = np.load('../data/fashion_data_complete.npz')

X_train = data['X_train']
y_train = data['y_train']
X_val = data['X_val']
y_val = data['y_val']

# Use only 10,000 training samples for tuning
X_train_small = X_train[:10000]
y_train_small = y_train[:10000]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 2. Hyperparameter Tuning
n_estimators_list = [50, 100, 200]
max_depth_list = [10, 20, None]

results = {
    'depth_10': [],
    'depth_20': [],
    'depth_None': []
}

print("Starting Random Forest Hyperparameter Tuning...")

for max_depth in max_depth_list:
    key = f"depth_{max_depth}"
    print(f"\n--- Testing max_depth = {max_depth} ---")
    
    for n_estimators in n_estimators_list:
        start = time.time()

        rf_model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42,
            n_jobs=-1
        )

        rf_model.fit(X_train_small, y_train_small)
        val_acc = rf_model.score(X_val, y_val)
        results[key].append(val_acc)

        end = time.time()
        print(f"n_estimators={n_estimators:3d} | Val Acc: {val_acc:.4f} | Time: {end-start:.2f}s")


# 3. Visualization for Report
plt.figure(figsize=(10, 6))

plt.plot(n_estimators_list, results['depth_10'], marker='o', label='max_depth=10')
plt.plot(n_estimators_list, results['depth_20'], marker='s', label='max_depth=20')
plt.plot(n_estimators_list, results['depth_None'], marker='^', label='max_depth=None')

plt.title('Random Forest Tuning: Number of Trees vs Max Depth')
plt.xlabel('Number of Trees (n_estimators)')
plt.ylabel('Validation Accuracy')
plt.xticks(n_estimators_list)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


# 4. Find Best Parameters
best_acc = -1
best_n_estimators = None
best_max_depth = None

for max_depth in max_depth_list:
    key = f"depth_{max_depth}"
    for i, n_estimators in enumerate(n_estimators_list):
        acc = results[key][i]
        if acc > best_acc:
            best_acc = acc
            best_n_estimators = n_estimators
            best_max_depth = max_depth

print(f"\nBest parameters found:")
print(f"n_estimators = {best_n_estimators}")
print(f"max_depth    = {best_max_depth}")
print(f"Best Val Acc = {best_acc:.4f}")

# 5. Train Final Model on Full Training Set
print("\nTraining final Random Forest model on full training set...")

final_rf = RandomForestClassifier(
    n_estimators=best_n_estimators,
    max_depth=best_max_depth,
    random_state=42,
    n_jobs=-1
)

final_rf.fit(X_train, y_train)

# Save model
joblib.dump(final_rf, 'random_forest_fashion_model.joblib')
print("Saved: random_forest_fashion_model.joblib")

### **Hyperparameter Tuning: Random Forest**

To optimize the performance of the Random Forest model, we conducted a grid search over two main hyperparameters: the number of trees (`n_estimators`) and the maximum tree depth (`max_depth`). The objective was to identify the configuration that achieves the best validation accuracy while maintaining a reasonable computational cost.

#### **1. Meaning of the Selected Hyperparameters**
* **`n_estimators`** is the number of decision trees in the forest. A larger number of trees usually makes the ensemble more stable and improves generalization, but it also increases training time.
* **`max_depth`** controls how deep each decision tree is allowed to grow. Smaller depth values make the trees simpler and faster, while larger depth values allow the model to learn more complex decision boundaries.

We selected these two hyperparameters because they are among the most influential settings in Random Forest and directly control the trade-off between model capacity, generalization ability, and computational cost.

#### **2. Analysis of the Number of Trees (`n_estimators`)**
We tested three values of `n_estimators`: {50, 100, 200}.  
* **Observation:** For all depth settings, increasing the number of trees generally improved validation accuracy. The improvement was more noticeable when increasing from 50 to 100 trees, while the gain from 100 to 200 trees was smaller.
* **Insight:** This behavior is expected in Random Forest. Adding more trees usually improves the stability and generalization ability of the ensemble because predictions are averaged over more diverse decision trees. However, after a certain point, the performance gain becomes marginal while the training time continues to increase.

#### **3. Analysis of Maximum Tree Depth (`max_depth`)**
We compared three depth settings: `max_depth = 10`, `max_depth = 20`, and `max_depth = None`.
* **Observation:** The shallowest configuration (`max_depth = 10`) consistently produced the lowest validation accuracy, around 83.37% to 83.60%. Increasing the depth to 20 significantly improved the performance, reaching up to 85.32%. The best results were obtained when no depth limit was imposed (`max_depth = None`), with the highest validation accuracy of 85.53%.
* **Insight:** A small maximum depth restricts the complexity of each tree, which may cause underfitting because the model cannot capture enough detail from the Fashion-MNIST images. Allowing deeper trees enables the model to learn more complex decision boundaries. In this experiment, the unrestricted-depth setting performed best, suggesting that the model benefited from higher flexibility without showing severe overfitting on the validation set.

#### **4. Why These Values Were Chosen**
The tested values were selected to cover a simple but meaningful range:
* `n_estimators = 50, 100, 200` allows us to observe how performance changes when the forest size increases from small to moderate.
* `max_depth = 10, 20, None` allows us to compare shallow trees, moderately deep trees, and fully grown trees.

This range is sufficient to reveal the main trend without making tuning too computationally expensive.

#### **5. Trade-off Between Accuracy and Training Time**
Training time increased as the number of trees increased. For example:
* At `max_depth = 10`, training time rose from 0.79s (50 trees) to 2.41s (200 trees).
* At `max_depth = 20`, training time rose from 1.05s to 3.64s.
* At `max_depth = None`, training time increased from 1.40s to 3.31s.

This indicates that while larger ensembles improve performance, they also require more computation. Therefore, the final choice should balance both predictive accuracy and efficiency.

#### **6. Optimal Configuration**
Based on the validation results, the best-performing Random Forest configuration is:
* **Number of Trees (`n_estimators`)**: 200
* **Maximum Tree Depth (`max_depth`)**: None
* **Validation Accuracy achieved**: **85.53%**

This configuration was selected for training the final Random Forest model on the full training set, as it achieved the highest validation accuracy among all tested settings.

### **Figure Analysis: Random Forest Tuning Curve**

The tuning plot illustrates the relationship between the number of trees (`n_estimators`) and validation accuracy under different values of maximum tree depth (`max_depth`).

* The **green curve** (`max_depth = None`) consistently achieved the highest validation accuracy across all values of `n_estimators`, indicating that deeper trees were more effective for this classification task.
* The **orange curve** (`max_depth = 20`) performed better than the shallow-tree setting but remained slightly below the unrestricted-depth configuration.
* The **blue curve** (`max_depth = 10`) produced the lowest accuracy in all cases, showing that overly shallow trees were not expressive enough to model the Fashion-MNIST dataset well.

Another clear pattern is that validation accuracy improved as the number of trees increased from 50 to 200, although the improvement became smaller at higher values. This suggests diminishing returns: adding more trees still helps, but the benefit gradually decreases compared to the extra training time.

Overall, the figure shows that both hyperparameters influence performance, but `max_depth` had the stronger effect in this experiment. The best point on the plot corresponds to `n_estimators = 200` and `max_depth = None`, which was selected as the final Random Forest configuration.

# MLP

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import joblib
from sklearn.neural_network import MLPClassifier

# 1. Hyperparameter candidates
hidden_sizes = [
    (256,),
    (128, 64),
    (256, 128),
    (128, 128),
    (256, 64),
]

alpha_values = [1e-5, 1e-4]
learning_rate_values = [0.0005, 0.001]

results = []
all_configs = []

print("Starting MLP Hyperparameter Tuning...")

# 2. Grid Search over hidden size, alpha, learning_rate_init
for hidden in hidden_sizes:
    for alpha in alpha_values:
        for lr in learning_rate_values:
            start = time.time()

            mlp_model = MLPClassifier(
                hidden_layer_sizes=hidden,
                activation='relu',
                solver='adam',
                alpha=alpha,
                batch_size=128,
                learning_rate_init=lr,
                max_iter=30,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=5,
                random_state=42
            )

            mlp_model.fit(X_train_small, y_train_small)
            val_acc = mlp_model.score(X_val, y_val)

            results.append(val_acc)
            all_configs.append((hidden, alpha, lr))

            end = time.time()
            print(
                f"Hidden={hidden} | alpha={alpha} | lr={lr} "
                f"| Val Acc: {val_acc:.4f} | Time: {end-start:.2f}s"
            )

# 3. Find best configuration
best_idx = np.argmax(results)
best_hidden, best_alpha, best_lr = all_configs[best_idx]

print("\nBest configuration found:")
print(f"hidden_layer_sizes = {best_hidden}")
print(f"alpha              = {best_alpha}")
print(f"learning_rate_init = {best_lr}")
print(f"best_val_acc       = {results[best_idx]:.4f}")

# 4. Visualization for Report
labels = [
    f"{hidden}\na={alpha}\nlr={lr}"
    for hidden, alpha, lr in all_configs
]

plt.figure(figsize=(14, 6))
plt.plot(range(len(results)), results, marker='o')
plt.xticks(range(len(labels)), labels, rotation=90)
plt.title('MLP Tuning: Hidden Size, Alpha, and Learning Rate')
plt.xlabel('Hyperparameter Configuration')
plt.ylabel('Validation Accuracy')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

# 5. Train Final Model on Full Training Set
final_mlp = MLPClassifier(
    hidden_layer_sizes=best_hidden,
    activation='relu',
    solver='adam',
    alpha=best_alpha,
    batch_size=128,
    learning_rate_init=best_lr,
    max_iter=30,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=5,
    random_state=42
)

print("Training final MLP model on full training set...")
final_mlp.fit(X_train, y_train)

# Save model
joblib.dump(final_mlp, 'mlp_fashion_model.joblib')
print("Saved: mlp_fashion_model.joblib")

### **Hyperparameter Tuning: Multi-layer Perceptron (MLP)**

To improve the performance of the MLP model, we performed a grid search over three important hyperparameters: the hidden layer configuration (`hidden_layer_sizes`), the regularization strength (`alpha`), and the initial learning rate (`learning_rate_init`). The objective was to identify the configuration that yields the highest validation accuracy while keeping the training process stable and efficient.

#### **1. Meaning of the Selected Hyperparameters**
* **`hidden_layer_sizes`** determines the number of hidden layers and the number of neurons in each layer. It controls the representational capacity of the network.
* **`alpha`** is the L2 regularization strength. It helps reduce overfitting by penalizing overly large weights.
* **`learning_rate_init`** is the initial step size used by the optimizer. It affects how quickly the model learns during training.

These hyperparameters were selected because they directly influence model complexity, generalization, and optimization behavior, making them the most important settings to tune for MLP in this task.

#### **2. Analysis of Hidden Layer Configuration**
We tested five hidden-layer settings: `(256,)`, `(128, 64)`, `(256, 128)`, `(128, 128)`, and `(256, 64)`.
* **Observation:** Among all tested architectures, `(256, 128)` produced the strongest overall results, including the highest validation accuracy of 86.95%. In contrast, `(128, 128)` generally performed worse, with validation accuracy staying around 85.47% to 85.87%.
* **Insight:** These results suggest that a larger and gradually decreasing architecture such as `(256, 128)` is more suitable for Fashion-MNIST. It provides enough capacity to capture complex image patterns while maintaining a structured compression of features across layers. Smaller or less effective architectures may not extract discriminative representations as well, which leads to lower validation performance.

#### **3. Analysis of Regularization Strength (`alpha`)**
We compared two values of the L2 regularization parameter: `1e-5` and `1e-4`.
* **Observation:** The effect of `alpha` was smaller than the effect of hidden-layer design, but it still influenced the final results. In the best-performing architecture `(256, 128)`, `alpha = 1e-5` achieved the strongest result when combined with `learning_rate_init = 0.001`.
* **Insight:** This suggests that a lighter regularization was slightly more suitable for the best MLP configuration, allowing the network to retain more flexibility while still generalizing well on the validation set.

#### **4. Analysis of Initial Learning Rate (`learning_rate_init`)**
We tested two values of the learning rate: `0.0005` and `0.001`.
* **Observation:** In several strong configurations, `learning_rate_init = 0.001` outperformed `0.0005`. The best overall result, 86.95%, was obtained with `learning_rate_init = 0.001`.
* **Insight:** A learning rate of `0.001` appears to provide a good balance between convergence speed and optimization stability. A smaller learning rate such as `0.0005` may require more training iterations to reach a similarly good solution, while `0.001` allows the optimizer to learn more efficiently within the limited number of epochs.

#### **5. Why These Values Were Chosen**
The tested values were chosen to cover a compact but meaningful search space:
* The hidden-layer settings include both single-layer and two-layer architectures, allowing us to compare different levels of model capacity.
* `alpha = 1e-5` and `1e-4` were selected to test two nearby regularization strengths without making the search too large.
* `learning_rate_init = 0.0005` and `0.001` were selected as practical values for stable training with the Adam optimizer.

This design allows us to evaluate the most important MLP behaviors while keeping the tuning process computationally manageable.

#### **6. Additional Fixed Settings**
Other training settings were kept fixed, including:
* `activation = relu`
* `solver = adam`
* `batch_size = 128`
* `max_iter = 30`
* `early_stopping = True`

These settings were chosen to ensure stable optimization and reduce unnecessary tuning complexity, so that the main comparison focuses on the most influential hyperparameters.

#### **7. Note on Convergence**
One configuration produced a `ConvergenceWarning`, indicating that the optimizer had not fully converged within 30 iterations. However, since the warning appeared only in a limited case and the validation accuracy remained meaningful, we still used the validation results to compare configurations fairly.

#### **8. Optimal Configuration**
Based on the validation results, the best-performing MLP configuration is:
* **Hidden Layer Sizes:** `(256, 128)`
* **Regularization Strength (`alpha`)**: `1e-5`
* **Initial Learning Rate (`learning_rate_init`)**: `0.001`
* **Validation Accuracy achieved:** **86.95%**

This configuration was selected to train the final MLP model on the full training set because it achieved the highest validation accuracy among all tested combinations.

### **Figure Analysis: MLP Tuning Curve**

The tuning plot presents the validation accuracy of the MLP model across different combinations of hidden-layer structure, regularization strength (`alpha`), and initial learning rate (`learning_rate_init`).

* A clear trend in the figure is that configurations based on **`(256, 128)`** occupy the highest region of the plot, showing that this architecture consistently outperformed the other tested hidden-layer settings.
* The configurations using **`(128, 128)`** tend to appear in the lower part of the curve, indicating weaker performance for this dataset.
* The effect of **`alpha`** is present but less dominant than the hidden-layer structure. In several cases, `alpha = 0.0001` produced slightly better validation accuracy than `1e-5`, suggesting that a moderate amount of regularization helped improve generalization.
* The effect of **`learning_rate_init`** is also noticeable. In many configurations, `0.001` led to better results than `0.0005`, especially for the strongest architectures.

Overall, the figure shows that the hidden-layer design had the largest influence on validation performance, while `alpha` and `learning_rate_init` provided smaller but still meaningful refinements. The best point on the plot corresponds to the configuration **`hidden_layer_sizes = (256, 128)`, `alpha = 0.0001`, and `learning_rate_init = 0.001`**, which was selected as the final MLP model.